In [1]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/08 16:39:21 WARN Utils: Your hostname, humacao, resolves to a loopback address: 127.0.1.1; using 192.168.68.83 instead (on interface wlp2s0)
26/03/08 16:39:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/08 16:39:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/08 16:39:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [2]:
DIRECTORY = "./data/broadcast_logs"

In [3]:
logs = (
    spark.read.csv(
        os.path.join(DIRECTORY, "BroadcastLogs_2018_Q3_M8.CSV"),
        sep="|",
        header=True,
        inferSchema=True,
        timestampFormat="yyyy-MM-dd",
    )
    .drop("BroadcastLogID", "SequenceNO")
    .withColumn(
        "duration_seconds",
        (
            F.col("Duration").substr(1, 2).cast("int") * 60 * 60
            + F.col("Duration").substr(4, 2).cast("int") * 60
            + F.col("Duration").substr(7, 2).cast("int")
        ),
    )
)

log_identifier = spark.read.csv(
    os.path.join(DIRECTORY, "ReferenceTables/LogIdentifier.csv"),
    sep="|",
    header=True,
    inferSchema=True,
)

cd_category = spark.read.csv(
    os.path.join(DIRECTORY, "ReferenceTables/CD_Category.csv"),
    sep="|",
    header=True,
    inferSchema=True,
).select(
    "CategoryID",
    "CategoryCD",
    F.col("EnglishDescription").alias("Category_Description"),
)

cd_program_class = spark.read.csv(
    os.path.join(DIRECTORY, "ReferenceTables/CD_ProgramClass.csv"),
    sep="|",
    header=True,
    inferSchema=True,
).select(
    "ProgramClassID",
    "ProgramClassCD",
    F.col("EnglishDescription").alias("ProgramClass_Description"),
)

In [4]:
logs_and_channels = logs.join(log_identifier, on="LogServiceID", how="inner")

In [8]:
full_log = logs_and_channels.join(cd_category, "CategoryID", how="left").join(
    cd_program_class, "ProgramClassID", how="left"
)

In [9]:
full_log.printSchema()

root
 |-- ProgramClassID: integer (nullable = true)
 |-- CategoryID: integer (nullable = true)
 |-- LogServiceID: integer (nullable = true)
 |-- LogDate: date (nullable = true)
 |-- AudienceTargetAgeID: integer (nullable = true)
 |-- AudienceTargetEthnicID: integer (nullable = true)
 |-- ClosedCaptionID: integer (nullable = true)
 |-- CountryOfOriginID: integer (nullable = true)
 |-- DubDramaCreditID: integer (nullable = true)
 |-- EthnicProgramID: integer (nullable = true)
 |-- ProductionSourceID: integer (nullable = true)
 |-- FilmClassificationID: integer (nullable = true)
 |-- ExhibitionID: integer (nullable = true)
 |-- Duration: string (nullable = true)
 |-- EndTime: string (nullable = true)
 |-- LogEntryDate: date (nullable = true)
 |-- ProductionNO: string (nullable = true)
 |-- ProgramTitle: string (nullable = true)
 |-- StartTime: string (nullable = true)
 |-- Subtitle: string (nullable = true)
 |-- NetworkAffiliationID: integer (nullable = true)
 |-- SpecialAttentionID: inte

In [17]:
# Calculate final result and show result
answer = full_log.groupby("LogIdentifierID").agg(
    F.sum(
        F.when(
            F.trim(F.col("ProgramClassCD")).isin(
                ["COM", "PRC", "PGI", "PRO", "LOC", "SPO", "MER", "SOL"]
            ),
            F.col("duration_seconds"),
        ).otherwise(0)
    ).alias("duration_commercial"),
    F.sum("duration_seconds").alias("duration_total"),
).withColumn(
    "commercial_ratio", F.col("duration_commercial") / F.col("duration_total")
).orderBy(
    "commercial_ratio", ascending=False
)

In [18]:
answer.show(
    10, False
)

[Stage 37:===========================================>              (6 + 2) / 8]

+---------------+-------------------+--------------+------------------+
|LogIdentifierID|duration_commercial|duration_total|commercial_ratio  |
+---------------+-------------------+--------------+------------------+
|CIMT           |19935              |19935         |1.0               |
|TELENO         |545255             |545255        |1.0               |
|MSET           |101670             |101670        |1.0               |
|TANG           |271468             |271468        |1.0               |
|TLNSP          |234455             |234455        |1.0               |
|TRN            |403                |403           |1.0               |
|HPITV          |403                |403           |1.0               |
|INVST          |623057             |633659        |0.9832686034602207|
|ZT�L�          |669624             |682023        |0.9818202611935375|
|CANALZ         |669624             |682023        |0.9818202611935375|
+---------------+-------------------+--------------+------------

In [19]:
call_signs = spark.read.csv(
    "data/broadcast_logs/Call_Signs.csv", header=True
).drop("UndertakingNo")

In [20]:
call_signs.printSchema()

root
 |-- LogIdentifierID: string (nullable = true)
 |-- Undertaking_Name: string (nullable = true)



In [21]:
new_answer = answer.join(call_signs, on="LogIdentifierID")

In [22]:
new_answer.show(10, False)

[Stage 45:===========================================>              (6 + 2) / 8]

+---------------+-------------------+--------------+-------------------+--------------------------------------------------------+
|LogIdentifierID|duration_commercial|duration_total|commercial_ratio   |Undertaking_Name                                        |
+---------------+-------------------+--------------+-------------------+--------------------------------------------------------+
|BRAVO          |701000             |3383060       |0.2072088582525879 |Bravo!                                                  |
|CI             |720130             |3401530       |0.2117076727237449 |Crime + Investigation (formerly Mystery)                |
|BOOK           |607620             |3292170       |0.18456519560047022|Book Television (formerly Book Television - The Channel)|
|BITE           |718540             |3411100       |0.2106475916859664 |Makeful TV (formerly BITE Television )                  |
|CBKT           |539627             |3221057       |0.1675310309628175 |Canadian Broadcast

In [25]:
# Load NYC Taxi data directly in PySpark
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"

In [26]:
df = spark.read.parquet(url)

UnsupportedOperationException: [FAILED_READ_FILE.UNSUPPORTED_FILE_SYSTEM] Encountered error while reading file https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet. The file system org.apache.hadoop.fs.http.HttpsFileSystem hasn't implemented listStatus. SQLSTATE: KD001

In [ ]:
df.printSchema()
df.show(5, False)